# Ecuadorian Vehicle Data Pipeline

## Imports and routes

In [1]:
import pandas as pd
import unicodedata
from notebookutils import mssparkutils

StatementMeta(sparkpool, 14, 2, Finished, Available, Finished, False)

## Raw variables

In [2]:
#Variables for raw
container_raw = "raw"
storageaccount = "asavehiclesccdls"
file_vehicles="datavehicles"
#Routes
raw_route_vehicles = f"abfss://{container_raw}@{storageaccount}.dfs.core.windows.net/{file_vehicles}"
#Files with route
files = mssparkutils.fs.ls(raw_route_vehicles)
#Dataframes
dataframes = []

StatementMeta(sparkpool, 14, 3, Finished, Available, Finished, False)

## Mapeo estructural de las columnas

In [3]:
columnmapping = {
    "codigo_vehiculo_1":"codigo_vehiculo",
    "codigo_de_vehiculo":"codigo_vehiculo",
    
    "sub_categoria_1":"categoria",
    "codigo_sub_categoria":"categoria",

    "forma_de_adquisicion":"tipo_transaccion",

    "fecha_proceso_(mm/dd/aa)":"fecha_proceso",
    "fecha_proceso_(dd/mm/aa)":"fecha_proceso",
    "fecha_proceso_(dd/mm/aaaa)":"fecha_proceso",
    "mes__registro_venta":"fecha_proceso",

    "fecha_compra_(mm/dd/aa)":"fecha_compra",
    "fecha_compra_(dd/mm/aa)":"fecha_compra",
    "fecha_compra_(dd/mm/aaaa)":"fecha_compra",
    "mes_adquisicion":"fecha_compra",

    "ano_modelo":"anio_modelo",

    "codigo_color_1":"color_1",

    "codigo_color_2":"color_2",

    "valor_avaluo":"avaluo",
    
    "descripcion_canton": "canton_nombre",
    "codigo_canton": "canton",
    "canton": "canton",

    "persona_natural_-_sociedad":"tipo_persona",
    "persona_natural_-_juridica":"tipo_persona"
}

StatementMeta(sparkpool, 14, 4, Finished, Available, Finished, False)

## Functions for CSV, no special characters and text cleaning

In [4]:
def read_csv(ruta):
    try:
        df=pd.read_csv(ruta,encoding="latin-1",sep=";",low_memory=False)
        return df
    except UnicodeDecodeError:
        print("File not read correctly")
def nospecialcharacter(texto):
    return''.join(
        c for c in unicodedata.normalize('NFD',texto)
        if unicodedata.category(c)!='Mn'
    )
def title_cleaning(col):
    col = nospecialcharacter(col)
    return(col.strip().lower().replace(" ","_"))

StatementMeta(sparkpool, 14, 5, Finished, Available, Finished, False)

## Files processing for column structure

In [5]:
for file in files:
    if file.path.endswith(".csv"):
        df = read_csv(file.path)
        df.columns = [title_cleaning(col) for col in df.columns]
        df=df.loc[:,~df.columns.str.startswith("unnamed")]
        df.rename(columns=columnmapping,inplace=True)
        if "tipo_persona" not in df.columns:
            df["tipo_persona"]=pd.NA
        anio = file.path.split("_")[-1].replace(".csv","")
        df["anio"]=int(anio)
        if int(anio) == 2017:
            for col in ["fecha_compra", "fecha_proceso"]:
                if col in df.columns:
                    mes = pd.to_numeric(df[col], errors="coerce")

                    df[col] = pd.to_datetime(
                        {
                            "year": int(anio),
                            "month": mes,
                            "day": 1
                        },
                        errors="coerce"
            )
        df = df.drop(columns=["canton_nombre"],errors="ignore")
        print(df.columns.tolist())
        dataframes.append(df)
        print(len(df.columns))
print(f"{len(dataframes)} archivos leidos correctamente")

StatementMeta(sparkpool, 14, 6, Finished, Available, Finished, False)

['codigo_vehiculo', 'categoria', 'marca', 'modelo', 'pais', 'anio_modelo', 'tipo', 'clase', 'sub_clase', 'cilindraje', 'tipo_combustible', 'tipo_servicio', 'color_1', 'color_2', 'tipo_transaccion', 'fecha_compra', 'canton', 'fecha_proceso', 'avaluo', 'tipo_persona', 'anio']
21
['categoria', 'codigo_vehiculo', 'tipo_transaccion', 'marca', 'modelo', 'pais', 'anio_modelo', 'clase', 'sub_clase', 'tipo', 'avaluo', 'fecha_proceso', 'tipo_servicio', 'cilindraje', 'tipo_combustible', 'fecha_compra', 'canton', 'color_1', 'color_2', 'tipo_persona', 'anio']
21
['categoria', 'codigo_vehiculo', 'tipo_transaccion', 'marca', 'modelo', 'pais', 'anio_modelo', 'clase', 'sub_clase', 'tipo', 'avaluo', 'fecha_proceso', 'tipo_servicio', 'cilindraje', 'tipo_combustible', 'fecha_compra', 'canton', 'color_1', 'color_2', 'tipo_persona', 'anio']
21
['categoria', 'codigo_vehiculo', 'tipo_transaccion', 'marca', 'modelo', 'pais', 'anio_modelo', 'clase', 'sub_clase', 'tipo', 'avaluo', 'fecha_proceso', 'tipo_servicio

## Standarization for columns to be used

In [6]:
mapeo_combustible = {
    "gas": "gas",
    "dual": "dual",
    "hibrido": "hibrido",
    "gasolina": "gasolina",
    "diesel": "diesel",
    "gas_natural_comprimido": "gas",
    "gas_licuado_petroleo": "gas",
    "dual_gas_gasolina": "dual",
    "electrico": "electrico",
    "hibrido_gasolina_baterias": "hibrido",
    "hibrido_diesel_baterias": "hibrido",
    "alcohol": "otros",
    "solar": "otros",
    "no_utiliza": "otros",
    "otros": "otros"
}
mapeo_colores = {
    "ama": "amarillo",
    "ana": "anaranjado",
    "aws": "sin especificar",
    "azu": "azul",
    "bla": "blanco",
    "caf": "cafe",
    "cam": "cafe",
    "cel": "celeste",
    "cob": "cobre",
    "cre": "crema",
    "cua": "sin especificar",
    "dor": "dorado",
    "fon": "sin especificar",
    "gry": "gris",
    "mor": "morado",
    "neg": "negro",
    "osc": "sin especificar",
    "pla": "plateado",
    "plo": "plomo",
    "roj": "rojo",
    "ros": "rosado",
    "sbm": "sin especificar",
    "sin": "sin especificar",
    "vin": "vino",
    "vrd": "verde",
    "zzz": "sin especificar"
}
meses = {
    "ene": "01",
    "feb": "02",
    "mar": "03",
    "abr": "04",
    "may": "05",
    "jun": "06",
    "jul": "07",
    "ago": "08",
    "sep": "09",
    "oct": "10",
    "nov": "11",
    "dic": "12"
}

StatementMeta(sparkpool, 14, 7, Finished, Available, Finished, False)

## Functions for text and date cleaning

In [7]:
def date_text_cleaning(col):
    col = (
        col.astype("string")
        .str.lower()
        .str.strip()
        .str.replace("ñ", "n")
        .str.replace("/", "-")
        .str.replace("Ð", "n", regex=False)
        .str.replace("?", "n", regex=False)
        .str.replace(r"\s+", " ", regex=True)
    )
    for mes,num in meses.items():
        col = col.str.replace(mes,num,regex=False)
    return pd.to_datetime(col,errors="coerce",dayfirst=True,format="mixed")
def text_cleaning(col):
        return(
        col.astype("string")
            .str.lower()
            .str.strip()
            .str.replace("ñ", "n")
            .str.replace("Ð", "n", regex=False)
            .str.replace("?", "n", regex=False)
            .str.replace(r"\s+", " ", regex=True)
    )

StatementMeta(sparkpool, 14, 8, Finished, Available, Finished, False)

## Column processing for register standarization

In [8]:
text_columns = ["clase","sub_clase","tipo_combustible","color_1","color_2"]
for i, df in enumerate(dataframes):
    df = df[df["codigo_vehiculo"].notna()].copy()
    for col in text_columns:
        if col in df.columns:
            df[col]=text_cleaning(df[col])
    if "codigo_vehiculo" in df.columns:
        df["codigo_vehiculo"] = (
            df["codigo_vehiculo"]
            .astype("string")
            .str.strip()
        )
    if df["anio"].iloc[0] != 2017:
        if "fecha_proceso" in df.columns:
            df["fecha_proceso"] = date_text_cleaning(df["fecha_proceso"])
        if "fecha_compra" in df.columns:
            df["fecha_compra"] = date_text_cleaning(df["fecha_compra"])
    if "sub_clase" in df.columns:
        df["sub_clase"]=df["sub_clase"].replace({
            "camion pequeno":"camion pequeno",
            "camion pequenno":"camion pequeno"
        })
    if "tipo_combustible" in df.columns:
        df["tipo_combustible"]=df["tipo_combustible"].map(mapeo_combustible).fillna("sin especificar")
    if "color_1" in df.columns:
        df["color_1"]=df["color_1"].map(mapeo_colores).fillna("sin especificar")
    if "color_2" in df.columns:
        df["color_2"]=df["color_2"].map(mapeo_colores).fillna("sin especificar")
    if "avaluo" in df.columns:
        df["avaluo"] = (
            df["avaluo"]
            .astype("string")
            .str.strip()
            .str.replace(",",".",regex=False)
            )
    dataframes[i] = df

StatementMeta(sparkpool, 14, 9, Finished, Available, Finished, False)

## Final dataframe

In [9]:
df_final = pd.concat(dataframes,ignore_index=True)
df_final = df_final.drop_duplicates()

df_final["fecha_compra"]=pd.to_datetime(
    df_final["fecha_compra"],
    errors="coerce",
    dayfirst=True
)
df_final["fecha_proceso"]=pd.to_datetime(
    df_final["fecha_proceso"],
    errors="coerce",
    dayfirst=True
)
df_final["avaluo"]=pd.to_numeric(
    df_final["avaluo"],
    errors="coerce"
)
df_final["tipo_persona"] = df_final["tipo_persona"].astype("string").fillna("sin especificar")
for col in df_final.columns:
    if df_final[col].dtype=="object":
        df_final[col] = df_final[col].astype("string")
print(df_final.dtypes)

StatementMeta(sparkpool, 14, 10, Finished, Available, Finished, False)

codigo_vehiculo     string[python]
categoria                    int64
marca               string[python]
modelo              string[python]
pais                string[python]
anio_modelo                  int64
tipo                string[python]
clase               string[python]
sub_clase           string[python]
cilindraje                   int64
tipo_combustible    string[python]
tipo_servicio       string[python]
color_1             string[python]
color_2             string[python]
tipo_transaccion    string[python]
fecha_compra        datetime64[ns]
canton                       int64
fecha_proceso       datetime64[ns]
avaluo                     Float64
tipo_persona        string[python]
anio                         int64
dtype: object


## Clean and curated variables

In [ ]:
container_clean = "clean"
container_curated = "curated"
storageaccount = "asavehiclesccdls"

clean_route = f"abfss://{container_clean}@{storageaccount}.dfs.core.windows.net/"
curated_route = f"abfss://{container_curated}@{storageaccount}.dfs.core.windows.net/"
raw_route = f"abfss://{container_raw}@{storageaccount}.dfs.core.windows.net/"
cantones_file = "tabla_homologacion.csv"
cantones = f"{raw_route}/{cantones_file}"

StatementMeta(sparkpool, 14, 11, Finished, Available, Finished, False)

## Merge with df final with 23 columns and cantones

In [11]:
csv_cantones = pd.read_csv(cantones,encoding="utf-8-sig")
# Cleaning
df_final["canton"]=df_final["canton"].astype("string").str.strip()
csv_cantones["canton"]=csv_cantones["canton"].astype("string").str.strip()
#Elimination of unnecesarry columns before merge
df_final = df_final.drop(
    columns=[
        "canton_nombre","provincia","canton_id_normalizado",
        "canton_nombre_x", "canton_nombre_y",
        "provincia_x", "provincia_y",
        "canton_id_normalizado_x", "canton_id_normalizado_y",
        "_merge"
    ],
    errors="ignore"
)
csv_cantones = csv_cantones.drop(columns=["_merge"], errors="ignore")

# ahora sí merge limpio
df_final = df_final.merge(
    csv_cantones,
    on="canton",
    how="left"
)

print(df_final.columns.tolist())

StatementMeta(sparkpool, 14, 12, Finished, Available, Finished, False)

['codigo_vehiculo', 'categoria', 'marca', 'modelo', 'pais', 'anio_modelo', 'tipo', 'clase', 'sub_clase', 'cilindraje', 'tipo_combustible', 'tipo_servicio', 'color_1', 'color_2', 'tipo_transaccion', 'fecha_compra', 'canton', 'fecha_proceso', 'avaluo', 'tipo_persona', 'anio', 'canton_nombre', 'provincia', 'provincia_mapa']


## Df curated

In [12]:
titles_columns = [
    "clase",
    "sub_clase",
    "tipo_combustible",
    "color_1",
    "color_2",
    "canton_nombre",
    "provincia"
]
for col in df_final.columns:
    if df_final[col].dtype == "object":
        df_final[col] = df_final[col].astype("string")

for col in titles_columns:
    if col in df_final.columns:
        df_final[col] = df_final[col].astype("string").str.title()
        
df_final["avaluo"]= pd.to_numeric(df_final["avaluo"],errors="coerce")
df_final["fecha_compra"]=pd.to_datetime(
    df_final["fecha_compra"],errors="coerce"
)
df_final["fecha_proceso"] = pd.to_datetime(
    df_final["fecha_proceso"], errors="coerce"
)
df_curated = df_final[[
    "codigo_vehiculo",
    "anio",
    "clase",
    "sub_clase",
    "tipo_combustible",
    "color_1",
    "color_2",
    "avaluo",
    "canton",
    "canton_nombre",
    "provincia",
    "fecha_proceso",
    "fecha_compra"
]].copy()

StatementMeta(sparkpool, 14, 13, Finished, Available, Finished, False)

# Tables for demand and snapshots

In [13]:
#Creation tables for demand as first register and snapshot or actual status
df_demand = (
    df_curated
    .sort_values(["anio","fecha_proceso"])
    .groupby("codigo_vehiculo")
    .first()
    .reset_index()
)
df_snapshot = (
    df_curated
    .sort_values(["anio","fecha_proceso"])
    .groupby("codigo_vehiculo")
    .last()
    .reset_index()
)

StatementMeta(sparkpool, 14, 14, Finished, Available, Finished, False)

## Transformation to parquet

In [15]:
spark_df_final = spark.createDataFrame(df_final)
spark_df_curated = spark.createDataFrame(df_curated)
spark_df_demand = spark.createDataFrame(df_demand)
spark_df_snapshot = spark.createDataFrame(df_snapshot)

# Guardar CLEAN
spark_df_final.write \
    .mode("overwrite") \
    .partitionBy("anio") \
    .parquet(clean_route + "vehicles_clean/")

# Guardar CURATED
spark_df_curated.write \
    .mode("overwrite") \
    .partitionBy("anio") \
    .parquet(curated_route + "vehicles_curated/")
# Guardar DEMAND
spark_df_demand.write \
    .mode("overwrite") \
    .partitionBy("anio") \
    .parquet(curated_route + "vehicles_demand/")
#Guardar SNAPSHOT
spark_df_snapshot.write \
    .mode("overwrite") \
    .parquet(curated_route + "vehicles_snapshot/")


StatementMeta(sparkpool, 12, 16, Finished, Available, Finished, False)

## Date Validations for 2017

In [41]:
df_final.loc[df_final["anio"] == 2017, "fecha_proceso"].dt.day.unique()

StatementMeta(sparkpool, 6, 42, Finished, Available, Finished, False)

array([1], dtype=int32)

In [42]:
df_final.loc[df_final["anio"] == 2017, "fecha_proceso"].dt.month.value_counts().sort_index()

StatementMeta(sparkpool, 6, 43, Finished, Available, Finished, False)

fecha_proceso
1     11762
2     12071
3     15470
4     15161
5     17836
6     18938
7     19923
8     18684
9     19234
10    19955
11    22428
12    21047
Name: count, dtype: int64

In [37]:
df_final.loc[df_final["anio"] == 2017, "fecha_proceso"].dt.day.unique()

StatementMeta(sparkpool, 6, 38, Finished, Available, Finished, False)

array([1], dtype=int32)

In [43]:
df_final.groupby("anio")["fecha_proceso"].apply(lambda x: x.isna().mean())

StatementMeta(sparkpool, 6, 44, Finished, Available, Finished, False)

anio
2017    0.0
2018    0.0
2019    0.0
2020    0.0
2021    0.0
2022    0.0
2023    0.0
2024    0.0
2025    0.0
2026    0.0
Name: fecha_proceso, dtype: float64

# Data analysis for change in preferences

In [26]:
conteo = df_final.groupby("codigo_vehiculo").size().reset_index(name="n_registros")
duplicados = conteo[conteo["n_registros"] > 1]

total_vehiculos = df_final["codigo_vehiculo"].nunique()
vehiculos_duplicados = len(duplicados)
registros_duplicados = duplicados["n_registros"].sum() - vehiculos_duplicados

print(f"Total vehículos únicos: {total_vehiculos:,}")
print(f"Vehículos con más de un registro: {vehiculos_duplicados:,}")
print(f"Registros duplicados generados: {registros_duplicados:,}")
# Primero filtrar solo los vehículos duplicados
codigos_duplicados = duplicados["codigo_vehiculo"].tolist()
df_dup = df_final[df_final["codigo_vehiculo"].isin(codigos_duplicados)].copy()

def contar_cambios(df, columna):
    """Cuenta vehículos donde el atributo cambió entre registros"""
    variaciones = (
        df.groupby("codigo_vehiculo")[columna]
        .nunique()
    )
    vehiculos_con_cambio = (variaciones > 1).sum()
    pct = (vehiculos_con_cambio / vehiculos_duplicados) * 100
    return vehiculos_con_cambio, round(pct, 2)

atributos = {
    "color_1": "Color Primario",
    "color_2": "Color Secundario", 
    "tipo_combustible": "Tipo de Combustible",
    "clase": "Clase de Vehículo",
    "sub_clase": "Sub Clase"
}

print("--- Cambios detectados en vehículos con registros repetidos ---")
resultados = []
for col, nombre in atributos.items():
    if col in df_dup.columns:
        n, pct = contar_cambios(df_dup, col) 
        resultados.append({
            "Atributo": nombre,
            "Vehículos con cambio": n,
            "% del total duplicados": pct
        })
        print(f"{nombre}: {n:,} vehículos ({pct}%)")

df_resultados = pd.DataFrame(resultados)
print("\n", df_resultados.to_string(index=False))

def analizar_color_secundario(df):
    sin_especificar = "Sin Especificar"
    
    def tipo_cambio(grupo):
        colores = grupo.sort_values("fecha_proceso")["color_2"].tolist()
        if len(set(colores)) == 1:
            return "sin_cambio"
        if colores[0] == sin_especificar and colores[-1] != sin_especificar:
            return "completado"
        return "cambio_real"
    
    resultado = (
        df.groupby("codigo_vehiculo")
        .apply(tipo_cambio, include_groups=False)
        .value_counts()
    )
    return resultado

if "color_2" in df_dup.columns:
    resultado_color2 = analizar_color_secundario(df_dup)
    print("\n--- Análisis Color Secundario ---")
    print(resultado_color2)

StatementMeta(sparkpool, 14, 27, Finished, Available, Finished, False)

Total vehículos únicos: 2,749,267
Vehículos con más de un registro: 301,912
Registros duplicados generados: 421,781
--- Cambios detectados en vehículos con registros repetidos ---
Color Primario: 601 vehículos (0.2%)
Color Secundario: 16,018 vehículos (5.31%)
Tipo de Combustible: 35 vehículos (0.01%)
Clase de Vehículo: 492 vehículos (0.16%)
Sub Clase: 1,536 vehículos (0.51%)

            Atributo  Vehículos con cambio  % del total duplicados
     Color Primario                   601                    0.20
   Color Secundario                 16018                    5.31
Tipo de Combustible                    35                    0.01
  Clase de Vehículo                   492                    0.16
          Sub Clase                  1536                    0.51

--- Análisis Color Secundario ---
sin_cambio     285894
completado      16009
cambio_real         9
Name: count, dtype: int64


In [25]:
# Verificar valores exactos en color_2 para vehículos duplicados
print("Valores únicos en color_2:")
print(df_dup["color_2"].unique()[:20])

print("\nValor más frecuente:")
print(df_dup["color_2"].value_counts().head(10))

# Ver un ejemplo de vehículo con cambio en color_2
ejemplo = df_dup[df_dup["codigo_vehiculo"].isin(
    df_dup.groupby("codigo_vehiculo")["color_2"]
    .nunique()
    .pipe(lambda x: x[x > 1])
    .index[:3]
)].sort_values(["codigo_vehiculo", "fecha_proceso"])

print("\nEjemplos de vehículos con cambio en color_2:")
print(ejemplo[["codigo_vehiculo", "fecha_proceso", "color_2"]])

StatementMeta(sparkpool, 14, 26, Finished, Available, Finished, False)

Valores únicos en color_2:
<StringArray>
['Sin Especificar',          'Blanco',        'Plateado',          'Dorado',
            'Azul',           'Plomo',            'Rojo',           'Negro',
           'Verde',        'Amarillo',            'Vino',      'Anaranjado',
           'Crema',            'Cafe',         'Celeste',          'Morado',
          'Rosado',           'Cobre']
Length: 18, dtype: string

Valor más frecuente:
color_2
Blanco             221154
Negro              129555
Sin Especificar     87769
Rojo                80592
Plomo               67864
Plateado            65526
Azul                29642
Vino                10478
Amarillo             8537
Verde                7592
Name: count, dtype: Int64

Ejemplos de vehículos con cambio en color_2:
        codigo_vehiculo fecha_proceso          color_2
2711451        10000203    2025-09-30  Sin Especificar
2711452        10000203    2025-11-30  Sin Especificar
3129114        10000203    2026-01-28           Blanco
2909